### <p style="text-align: center;"> **Name:** Shahzaib Khan &ensp;&ensp;&ensp;&ensp;&ensp;&ensp; **Roll No:** 19K-0273 &ensp;&ensp;&ensp;&ensp;&ensp;&ensp; **Section:** BCS-8B</p>

In [86]:
import numpy as np
import pandas as pd

# Generate a sample dataset
data = {
    'Refund': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No'],
    'Status': ['Single', 'Married', 'Single', 'Married', 'Divorced', 'Married', 'Divorced', 'Single', 'Married', 'Single'],
    'Income': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
    'Evade': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
df.head()

,Refund,Status,Income,Evade
0,Yes,Single,125,No
1,No,Married,100,No
2,No,Single,70,No
3,Yes,Married,120,No
4,No,Divorced,95,Yes


In [87]:
# Map categorical variables
df['Refund'] = df['Refund'].map({'Yes': 1, 'No': 0})
df['Status'] = df['Status'].map({'Single': 0, 'Married': 1, 'Divorced': 2})
df['Evade'] = df['Evade'].map({'Yes': 1, 'No': 0})

df.head()

,Refund,Status,Income,Evade
0,1,0,125,0
1,0,1,100,0
2,0,0,70,0
3,1,1,120,0
4,0,2,95,1


In [88]:
# Prior probabilities
class_counts = df['Evade'].value_counts()
total_samples = len(df)

prior_probs = {cls: count / total_samples for cls, count in class_counts.items()}
print("Prior probabilities:", prior_probs)

Prior probabilities: {0: 0.7, 1: 0.3}


In [89]:
# Categorical probabilities
def calc_cat_probs(feature, class_label, df):
    class_subset = df[df['Evade'] == class_label]
    counts = class_subset[feature].value_counts()
    total = len(class_subset)
    return counts / total

cat_probs = {}
for feature in ['Refund', 'Status']:
    cat_probs[feature] = {cls: calc_cat_probs(feature, cls, df) for cls in prior_probs.keys()}

# Print to verify
print("Conditional probabilities (Categorical):")
for feature in cat_probs:
    for cls in cat_probs[feature]:
        print(f"P({feature} | Evade={'Yes' if cls == 1 else 'No'}):\n{cat_probs[feature][cls]}")
# print(cat_probs)

Conditional probabilities (Categorical):
P(Refund | Evade=No):
Refund
0    0.571429
1    0.428571
Name: count, dtype: float64
P(Refund | Evade=Yes):
Refund
0    1.0
Name: count, dtype: float64
P(Status | Evade=No):
Status
1    0.571429
0    0.285714
2    0.142857
Name: count, dtype: float64
P(Status | Evade=Yes):
Status
0    0.666667
2    0.333333
Name: count, dtype: float64


In [90]:
# Income statistics
income_stats = df.groupby('Evade')['Income'].agg(['mean', 'var'])
print("Income statistics:\n", income_stats)

def gauss_prob(x, mean, variance):
    return (1 / np.sqrt(2 * np.pi * variance)) * np.exp(-((x - mean) ** 2) / (2 * variance))

Income statistics:
         mean     var
Evade               
0      110.0  2975.0
1       90.0    25.0


In [91]:
# Prediction
def predict(X):
    probs = {}
    for cls in prior_probs.keys():
        probs[cls] = np.log(prior_probs[cls])  # Log of prior

        # Categorical features
        for feature in ['Refund', 'Status']:
            value = X[feature]
            prob = cat_probs[feature][cls].get(value, 0)  # Use 0 if value not present
            probs[cls] += np.log(prob if prob > 0 else 1e-6)  # Avoid log(0)

        # Income (Gaussian)
        mean = income_stats.loc[cls, 'mean']
        variance = income_stats.loc[cls, 'var']
        probs[cls] += np.log(gauss_prob(X['Income'], mean, variance))

    return max(probs, key=probs.get)

In [92]:
# Test cases
X1 = {'Refund': 1, 'Status': 2, 'Income': 90}  # Refund=Yes, Status=Divorced, Income=90K
X2 = {'Refund': 0, 'Status': 1, 'Income': 60}  # Refund=No, Status=Married, Income=60K

p1 = predict(X1)
p2 = predict(X2)
print(f"Prediction for X1: {'Yes' if p1 == 1 else 'No'}")
print(f"Prediction for X2: {'Yes' if p2 == 1 else 'No'}")

Prediction for X1: No
Prediction for X2: No
